# Lemon PYNQ-Z1 AD9102 + AD9226 test

Run cells in order. The AD9102 uses a 180 MHz DAC clock.

In [ ]:
# Cell 1: load the overlay and create fixed-address handles.
from pynq import Overlay, MMIO
import matplotlib.pyplot as plt
import numpy as np

from lemon_pynqz1_ad9102 import AD9102
from lemon_pynqz1_capture import find_dma, run_dma_capture

BITFILE = 'base_add.bit'
overlay = Overlay(BITFILE)
led_ip = MMIO(0x40000000, 0x1000)
adc_ip = MMIO(0x40001000, 0x1000)
dds_ip = MMIO(0x40002000, 0x1000)
dma = find_dma(overlay)
dds = AD9102(0x40002000)

print('Overlay loaded:', BITFILE)
print('LED  MMIO: 0x40000000')
print('ADC  MMIO: 0x40001000')
print('DDS  MMIO: 0x40002000')
print('DDS status:', dds.status())

In [ ]:
# Cell 2: reset AD9102 and start a conservative 1 MHz sine wave.
result = dds.initialize(freq_hz=1_000_000, amplitude=0x400)
print(result)
print('DDS status:', dds.status())

In [ ]:
# Cell 3: verify the new 180 MHz frequency calculation at 60 MHz.
result = dds.configure_sine(freq_hz=60_000_000, amplitude=0x400)
print('requested = %d Hz' % result['requested_hz'])
print('FTW       = 0x%06X' % result['ftw'])
print('actual    = %.3f Hz' % result['actual_hz'])
print('Measure the output with a suitable high-bandwidth oscilloscope.')

In [ ]:
# Cell 4: load an SRAM arbitrary waveform using the STM32-compatible format.
# AD9102 SRAM words are signed 12-bit samples, left justified by 2 bits.
points = 1024
phase = np.arange(points)
triangle = np.where(
    phase < points // 2,
    -1024 + (phase * 2047) // (points // 2 - 1),
    1023 - ((phase - points // 2) * 2047) // (points // 2 - 1),
).astype(np.int16)

def show_progress(done, total):
    print('SRAM %4d / %4d' % (done, total), end='\r' if done < total else '\n')

result = dds.configure_arbitrary(
    triangle,
    freq_hz=1_000_000,
    amplitude=0x400,
    progress=show_progress,
)
print(result)

In [ ]:
# Cell 5: run DDS and real ADC capture together.
# Connect the DDS analog output to the ADC input before running this cell.
dds.configure_sine(freq_hz=1_000_000, amplitude=0x400)
raw, ch0, ch1 = run_dma_capture(
    adc_ip,
    dma,
    sample_count=32768,
    adc_half_period=2,
    decimation=1,
    capture_mode=1,
    sample_delay=1,
    verbose=True,
)

count = 1000
t_us = np.arange(count) / 31_250_000.0 * 1e6
plt.figure(figsize=(12, 4))
plt.plot(t_us, ch0[:count], label='ADC A')
plt.plot(t_us, ch1[:count], label='ADC B', alpha=0.7)
plt.xlabel('Time (us)')
plt.ylabel('ADC code')
plt.grid(True)
plt.legend()
plt.show()